[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradypuz/TorchCode-web-ui/blob/master/solutions/43_grpo_complete_loss_solution.ipynb)

# Solution: GRPO Loss - Complete

Reference implementation of the clipped, response-normalized GRPO objective with a sampled-token reference-policy KL penalty and safe padding handling.


In [ ]:
# Install the source version that includes this new exercise in Colab.
# No-op in JupyterLab/Docker, where the repository package is already available.
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q git+https://github.com/bradypuz/TorchCode-web-ui.git@master')
except ImportError:
    pass


In [ ]:
import torch
from torch import Tensor


In [ ]:
# ✅ SOLUTION

def grpo_complete_loss(
    policy_logps: Tensor,
    old_logps: Tensor,
    ref_logps: Tensor,
    rewards: Tensor,
    token_mask: Tensor,
    beta: float = 0.04,
    epsilon: float = 0.2,
) -> Tensor:
    """Clipped GRPO loss with a reference-policy KL penalty and response masking.

    policy_logps: (B, G, T) current log-probs of the sampled response tokens
    old_logps: (B, G, T) rollout-policy log-probs of those same tokens
    ref_logps: (B, G, T) reference-policy log-probs of those same tokens
    rewards: (B, G) scalar rewards for G responses to each of B prompts
    token_mask: (B, G, T) bool or 0/1 mask of valid response tokens
    returns: scalar loss; gradients flow only through valid policy_logps
    """
    # Sanitize padding before exponentiation; padding values may be NaN or Inf.
    valid = token_mask.detach().bool()
    current = torch.where(valid, policy_logps, 0.0)
    old = torch.where(valid, old_logps.detach(), 0.0)
    ref = torch.where(valid, ref_logps.detach(), 0.0)

    # Normalize rewards independently within each prompt's response group.
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False).clamp_min(1e-6)
    advantages = ((rewards - mean) / std).detach().unsqueeze(-1)

    # PPO-style clipping must use the minimum for both advantage signs.
    ratio = torch.exp(current - old)
    unclipped = ratio * advantages
    clipped = ratio.clamp(1 - epsilon, 1 + epsilon) * advantages
    policy_loss = -torch.minimum(unclipped, clipped)

    # Sampled-token KL estimator; expm1 is more accurate near log_ratio = 0.
    log_ratio = ref - current
    kl = torch.expm1(log_ratio) - log_ratio
    token_losses = policy_loss + beta * kl

    # Give each response equal weight, regardless of its valid token count.
    mask = valid.to(policy_logps.dtype)
    per_response_loss = (
        (token_losses * mask).sum(dim=-1)
        / mask.sum(dim=-1).clamp_min(1)
    )
    return per_response_loss.mean()


In [ ]:
# Demo
old_logps = torch.full((1, 2, 2), -2.0)
ratios = torch.tensor([[[1.4, 1.0], [0.6, 1.0]]])
policy_logps = (old_logps + ratios.log()).requires_grad_()
ref_logps = old_logps.clone()
rewards = torch.tensor([[1.0, -1.0]])
token_mask = torch.tensor([[[True, True], [True, False]]])
loss = grpo_complete_loss(policy_logps, old_logps, ref_logps, rewards, token_mask)
print('Loss:', loss.item())
# Expected: approximately -0.146376 with beta=0.04 and epsilon=0.2


In [ ]:
from torch_judge import check
check('grpo_complete_loss')
